# LDA Topic Discovery for Cluster 6: Work, Jobs & Workplace Life

Reads a CSV file, filters to **Cluster 6** subreddits (Work, jobs, workplace life & worker communities),
preprocesses English text with custom stopwords, runs Latent Dirichlet Allocation
using collapsed Gibbs sampling, and outputs discovered topics with document assignments.

**Pipeline:**
1. Load data and filter to cluster 6 subreddits
2. Preprocess text with custom stopwords (matching BERTopic notebook)
3. Build document-term matrix with bigrams
4. Fit LDA model
5. Display and save results

In [ ]:
import os
import logging
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import lda

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

## Configuration

In [ ]:
# ===== CONFIGURATION =====
CSV_FILE = "reddit_cleaned_01_13_first10.csv"   # Change to full dataset when ready
CLUSTER_PATH = "subreddit_cluster_summary_k20.csv"
TARGET_CLUSTER = 6          # Which cluster to explore (change to 0-19 for other clusters)
TEXT_COLUMN = "merged_text"  # Column with text to model

N_TOPICS = 10
N_ITER = 500
N_TOP_WORDS = 10
MAX_FEATURES = None
MIN_DF = 2
MAX_DF = 0.95
ALPHA = 0.1
ETA = 0.01
RANDOM_STATE = 42

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PREFIX = f"{OUTPUT_DIR}/cluster{TARGET_CLUSTER}_lda"
# =========================

## 1. Load Data & Filter Cluster 6

In [ ]:
# Load cluster summary and extract target cluster subreddits
cluster_df = pd.read_csv(CLUSTER_PATH)
c = cluster_df[cluster_df["cluster"] == TARGET_CLUSTER]

print(f"Cluster {TARGET_CLUSTER}: {c['Name by LLM'].values[0]}")
print(f"Top words: {c['top_words'].values[0]}")
print(f"Expected subreddits: {c['n_subreddits'].values[0]}, expected rows: {c['n_rows'].values[0]}")

# Parse subreddits (semicolon-separated)
cluster_subreddits = [s.strip() for s in c["subreddits"].values[0].split(";")]
print(f"\nSubreddits in cluster {TARGET_CLUSTER} ({len(cluster_subreddits)}):")
print(cluster_subreddits)

In [ ]:
# Load the full dataset and filter to target cluster
df_all = pd.read_csv(CSV_FILE)
print(f"Total rows in dataset: {len(df_all)}")

df = df_all[df_all["subreddit"].isin(cluster_subreddits)].copy()
print(f"Rows matching cluster {TARGET_CLUSTER} subreddits: {len(df)}")

if len(df) < 20:
    print(f"\nWARNING: Only {len(df)} documents found for cluster {TARGET_CLUSTER}.")
    print("   LDA needs more documents for meaningful topic discovery.")
    print("   Please update CSV_FILE to your full dataset CSV.")

# Drop rows with missing or empty text
mask = df[TEXT_COLUMN].notna() & (df[TEXT_COLUMN].astype(str).str.strip() != "")
n_dropped = (~mask).sum()
if n_dropped > 0:
    print(f"Dropped {n_dropped} rows with empty text")
df = df[mask].reset_index(drop=True)
documents = df[TEXT_COLUMN].astype(str).tolist()

print(f"\nDocuments ready: {len(documents)}")
print(f"Subreddits found: {sorted(df['subreddit'].unique().tolist())}")

## 2. Build Document-Term Matrix (with Custom Stopwords)

Custom stopwords match the BERTopic notebook to reduce noise words from Reddit posts.

In [ ]:
# Custom stopwords: domain-specific noise words (matching BERTopic notebook)
custom_stopwords = [
    # Reddit-specific noise
    "like", "just", "got", "get", "going", "would", "could", "really",
    "also", "know", "think", "want", "even", "still", "much", "thing",
    "things", "way", "make", "made", "said", "one", "people", "time",
    "don", "didn", "doesn", "isn", "wasn", "won", "wouldn", "couldn",
    "shouldn", "hasn", "hadn", "aren", "weren", "ll", "ve",
    # Conversational noise commonly seen in Reddit posts/comments
    "yeah", "okay", "lol", "lmao", "edit", "update", "deleted",
    "comment", "comments", "post", "posted", "thread", "subreddit",
    "reddit", "op", "username",
    # Relationship/story noise often appearing in work-related posts
    "sister", "sisters", "brother", "husband", "wife", "friend",
    "mom", "dad", "family",
]

# Merge with sklearn's English stopwords
all_stopwords = list(ENGLISH_STOP_WORDS.union(custom_stopwords))
print(f"Total stopwords: {len(all_stopwords)}")

vectorizer = CountVectorizer(
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    max_df=MAX_DF,
    stop_words=all_stopwords,
    ngram_range=(1, 2),        # Unigrams + bigrams (matching BERTopic notebook)
)
dtm = vectorizer.fit_transform(documents)
vocab = vectorizer.get_feature_names_out()

# Remove all-zero rows
row_sums = np.array(dtm.sum(axis=1)).flatten()
nonzero_mask = row_sums > 0
n_empty = (~nonzero_mask).sum()
if n_empty > 0:
    print(f"{n_empty} documents have no terms after vocabulary filtering")
dtm_filtered = dtm[nonzero_mask]

print(f"Corpus: {dtm_filtered.shape[0]} documents, {dtm_filtered.shape[1]} terms")

## 3. Run LDA

In [ ]:
model = lda.LDA(
    n_topics=N_TOPICS,
    n_iter=N_ITER,
    alpha=ALPHA,
    eta=ETA,
    random_state=RANDOM_STATE,
)
model.fit(dtm_filtered)

## 4. Display Topics

In [ ]:
print(f"{'='*60}")
print(f"Discovered {model.n_topics} Topics")
print(f"{'='*60}\n")

for i, topic_dist in enumerate(model.topic_word_):
    top_indices = np.argsort(topic_dist)[::-1][:N_TOP_WORDS]
    top_words = [(vocab[j], topic_dist[j]) for j in top_indices]
    words_str = ", ".join(f"{w} ({p:.4f})" for w, p in top_words)
    print(f"Topic {i}: {words_str}")

## 5. Assign Topics to Documents

In [ ]:
dominant_topics = np.full(len(df), -1, dtype=int)
topic_probs = np.full(len(df), np.nan)

doc_topic = model.doc_topic_
valid_indices = np.where(nonzero_mask)[0]
for local_idx, global_idx in enumerate(valid_indices):
    dominant_topics[global_idx] = np.argmax(doc_topic[local_idx])
    topic_probs[global_idx] = doc_topic[local_idx].max()

df["dominant_topic"] = dominant_topics
df["topic_probability"] = topic_probs

df[[TEXT_COLUMN, "dominant_topic", "topic_probability"]].head(10)

## 6. Save Results

In [ ]:
# Save document assignments
assignments_path = f"{OUTPUT_PREFIX}_assignments.csv"
df.to_csv(assignments_path, index=False)
print(f"Document assignments saved to: {assignments_path}")

# Save topic descriptions
topic_rows = []
for i, topic_dist in enumerate(model.topic_word_):
    top_indices = np.argsort(topic_dist)[::-1][:N_TOP_WORDS]
    row = {"topic_id": i}
    for rank, j in enumerate(top_indices):
        row[f"word_{rank+1}"] = vocab[j]
        row[f"prob_{rank+1}"] = round(float(topic_dist[j]), 6)
    topic_rows.append(row)

topics_path = f"{OUTPUT_PREFIX}_topics.csv"
pd.DataFrame(topic_rows).to_csv(topics_path, index=False)
print(f"Topic descriptions saved to: {topics_path}")
print("\nDone!")